<a href="https://colab.research.google.com/github/emmanuelokellootieno-afk/nairobi-urban-expansion-geoai/blob/main/CAProjectionCloud_Revised1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# nrb_config.py  —  SHARED CONFIGURATION
# Single source of truth for every folder name, export description,
# filename, and path used across all six parts.
#
# HOW TO USE:
#   Copy this block to the TOP of each part, or run it as its own cell.
#   Every part imports from this namespace — change names here only.
#
# NAMING CONVENTION (conflict-free):
#   • Drive folder : NrbCA_v3/
#   • Masks folder : NrbCA_v3/Masks/
#   • GEE export descriptions prefix: nrbca3_
#   • Local .npy/.pkl/.json files prefix: nrbca3_
#   • TIF filenames match GEE export descriptions exactly
#
# This eliminates conflicts with:
#   Nairobi_CA_Final/           (previous run)
#   Nairobi_DigitalTwin_2030/   (earlier run)
#   Nairobi_DigitalTwin/        (earliest run)
# =============================================================================

import os
from datetime import datetime

# ── Version tag — increment if you want a completely fresh run ────────────
VER = 'v3'

# ── Drive folder layout ───────────────────────────────────────────────────
# Top-level folder in MyDrive
DRIVE_BASE    = f'NrbCA_{VER}'
# Subfolder for annual urban mask TIFs
DRIVE_MASKS   = f'{DRIVE_BASE}/Masks'
# Local Colab paths
BASE_PATH     = f'/content/drive/MyDrive/{DRIVE_BASE}/'
MASKS_PATH    = f'/content/drive/MyDrive/{DRIVE_MASKS}/'

# ── GEE export description prefix ────────────────────────────────────────
# GEE uses the description as the filename, so prefix ensures uniqueness
P = f'nrbca{VER}_'    # e.g. 'nrbcav3_'

# ── GEE export descriptions (= filename on Drive without .tif/.csv) ───────
DESC_MASK        = lambda y: f'{P}mask_{y}'          # e.g. nrbcav3_mask_2017
DESC_EMBED_2025  = f'{P}embed_2025'                  # nrbcav3_embed_2025
DESC_SLOPE       = f'{P}slope'                       # nrbcav3_slope
DESC_EXCLUSION   = f'{P}exclusion'                   # nrbcav3_exclusion
DESC_TRANSITIONS = f'{P}transitions'                 # nrbcav3_transitions (CSV)

# ── Exact TIF filenames on Drive (GEE appends .tif automatically) ─────────
TIF_MASK         = lambda y: MASKS_PATH + f'{DESC_MASK(y)}.tif'
TIF_EMBED_2025   = BASE_PATH + f'{DESC_EMBED_2025}.tif'
TIF_SLOPE        = BASE_PATH + f'{DESC_SLOPE}.tif'
TIF_EXCLUSION    = BASE_PATH + f'{DESC_EXCLUSION}.tif'

# ── Local numpy / pickle / json filenames ─────────────────────────────────
NPY_URBAN_2025   = BASE_PATH + f'{P}urban2025.npy'
NPY_URBAN_2020   = BASE_PATH + f'{P}urban2020.npy'
NPY_SLOPE        = BASE_PATH + f'{P}slope.npy'
NPY_EXCLUSION    = BASE_PATH + f'{P}exclusion.npy'
NPY_SUIT_2025    = BASE_PATH + f'{P}suit2025.npy'
NPY_SUIT_2020    = BASE_PATH + f'{P}suit2020.npy'

PKL_PCA          = BASE_PATH + f'{P}pca.pkl'
PKL_RF           = BASE_PATH + f'{P}rf.pkl'

CSV_TRANSITIONS  = BASE_PATH + f'{DESC_TRANSITIONS}.csv'

JSON_META        = BASE_PATH + f'{P}meta.json'
JSON_TIMESERIES  = BASE_PATH + f'{P}timeseries.json'
JSON_VALIDATION  = BASE_PATH + f'{P}validation.json'

# ── Output TIF filenames (simulation results) ─────────────────────────────
TIF_HINDCAST     = BASE_PATH + f'{P}hindcast_2020_2025.tif'
TIF_BAU_2030     = BASE_PATH + f'{P}proj_BAU_2030.tif'
TIF_COMPACT_2030 = BASE_PATH + f'{P}proj_compact_2030.tif'
TIF_CLIMATE_2030 = BASE_PATH + f'{P}proj_climate_2030.tif'
GEOJSON_SUBCTY   = BASE_PATH + f'{P}subcounty_2030.geojson'

# ── Figure filenames ──────────────────────────────────────────────────────
FIG_TRAJECTORIES = BASE_PATH + f'{P}fig1_trajectories.png'
FIG_SCENARIO_MAP = BASE_PATH + f'{P}fig2_scenario_map.png'
FIG_VALIDATION   = BASE_PATH + f'{P}fig3_validation.png'
FIG_SUBCOUNTY    = BASE_PATH + f'{P}fig4_subcounty.png'
FIG_LOADINGS_PC2 = BASE_PATH + f'{P}fig_loadings_PC2.png'
FIG_LOADINGS_PC3 = BASE_PATH + f'{P}fig_loadings_PC3.png'
FIG_PCA_HEATMAP  = BASE_PATH + f'{P}fig_pca_heatmap.png'
FIG_RF_IMP       = BASE_PATH + f'{P}fig_rf_importance.png'

# ── Model constants ───────────────────────────────────────────────────────
PIXEL_KM2        = 0.0001    # 10 m × 10 m = 0.0001 km²
ALL_YEARS        = list(range(2017, 2026))   # 2017–2025 inclusive
HOLDOUT_YEAR     = 2020      # excluded from RF training; used for FoM
N_PER_CLASS      = 8000      # balanced samples per year-pair per class
GEE_PROJECT      = 'ee-emmanuelokellootieno'
AOI_ASSET        = 'projects/ee-emmanuelokellootieno/assets/Nairobi_Metropoli'
TRAINING_ASSET   = 'projects/ee-emmanuelokellootieno/assets/Nairobi_Trg3'
EMB_COLLECTION   = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
CRS_EXPORT       = 'EPSG:32737'

# ── Create Drive folders ──────────────────────────────────────────────────
def make_folders():
    os.makedirs(BASE_PATH,  exist_ok=True)
    os.makedirs(MASKS_PATH, exist_ok=True)
    print(f"Drive folder : MyDrive/{DRIVE_BASE}/")
    print(f"Masks folder : MyDrive/{DRIVE_MASKS}/")

if __name__ == '__main__':
    make_folders()
    print(f"\nConfig loaded — version {VER} — {datetime.now()}")
    print(f"All exports prefixed with: '{P}'")
    print(f"\nSample names:")
    print(f"  Annual mask 2021 desc : {DESC_MASK(2021)}")
    print(f"  Annual mask 2021 TIF  : {TIF_MASK(2021)}")
    print(f"  Embedding TIF         : {TIF_EMBED_2025}")
    print(f"  Transition CSV        : {CSV_TRANSITIONS}")
    print(f"  RF model pkl          : {PKL_RF}")
    print(f"  Suitability 2025 npy  : {NPY_SUIT_2025}")
    print(f"  BAU 2030 TIF          : {TIF_BAU_2030}")


Drive folder : MyDrive/NrbCA_v3/
Masks folder : MyDrive/NrbCA_v3/Masks/

Config loaded — version v3 — 2026-04-12 16:35:43.267724
All exports prefixed with: 'nrbcav3_'

Sample names:
  Annual mask 2021 desc : nrbcav3_mask_2021
  Annual mask 2021 TIF  : /content/drive/MyDrive/NrbCA_v3/Masks/nrbcav3_mask_2021.tif
  Embedding TIF         : /content/drive/MyDrive/NrbCA_v3/nrbcav3_embed_2025.tif
  Transition CSV        : /content/drive/MyDrive/NrbCA_v3/nrbcav3_transitions.csv
  RF model pkl          : /content/drive/MyDrive/NrbCA_v3/nrbcav3_rf.pkl
  Suitability 2025 npy  : /content/drive/MyDrive/NrbCA_v3/nrbcav3_suit2025.npy
  BAU 2030 TIF          : /content/drive/MyDrive/NrbCA_v3/nrbcav3_proj_BAU_2030.tif


In [ ]:
# =============================================================================
# NAIROBI URBAN GROWTH MODEL — AlphaEarth + CA   [v3]
# PART 1 OF 6: SETUP, GEE CLASSIFIER & URBAN MASK EXPORTS
#
# What this does:
#   • Installs libraries and authenticates GEE
#   • Trains RF classifier on 2025 AlphaEarth embeddings
#   • Generates annual urban masks 2017–2025 in GEE memory
#   • Submits export tasks to Drive (masks, embedding TIF, slope, exclusion)
#
# After running:
#   Go to https://code.earthengine.google.com/ → Tasks tab
#   Wait for ALL tasks to show green tick before running Part 2.
#   Typical wait: 15–45 min.
# =============================================================================

# ── Installs ──────────────────────────────────────────────────────────────
!pip install -U geemap earthengine-api scikit-learn rasterio \
    geopandas shapely scipy scikit-image rasterstats joblib --quiet

# ── Imports ───────────────────────────────────────────────────────────────
import ee
import os
import gc
import numpy as np
from datetime import datetime
from google.colab import drive

# Ensure the mount point is clean before attempting to mount
!rm -rf /content/drive/*
drive.mount('/content/drive', force_remount=True)

# ── Earth Engine auth ─────────────────────────────────────────────────────
ee.Authenticate()

# =============================================================================
# CONFIGURATION  —  single source of truth for every name in this project
# Paste this block unchanged into every part.
# =============================================================================
VER           = 'v3'
P             = f'nrbca{VER}_'          # all export descriptions + filenames

DRIVE_BASE    = f'NrbCA_{VER}'          # MyDrive/NrbCA_v3/
DRIVE_MASKS   = f'{DRIVE_BASE}/Masks'   # MyDrive/NrbCA_v3/Masks/
BASE_PATH     = f'/content/drive/MyDrive/{DRIVE_BASE}/'
MASKS_PATH    = f'/content/drive/MyDrive/{DRIVE_MASKS}/'

DESC_MASK        = lambda y: f'{P}mask_{y}'
DESC_EMBED_2025  = f'{P}embed_2025'
DESC_SLOPE       = f'{P}slope'
DESC_EXCLUSION   = f'{P}exclusion'
DESC_TRANSITIONS = f'{P}transitions'

TIF_MASK         = lambda y: MASKS_PATH + f'{DESC_MASK(y)}.tif'
TIF_EMBED_2025   = BASE_PATH + f'{DESC_EMBED_2025}.tif'
TIF_SLOPE        = BASE_PATH + f'{DESC_SLOPE}.tif'
TIF_EXCLUSION    = BASE_PATH + f'{DESC_EXCLUSION}.tif'
CSV_TRANSITIONS  = BASE_PATH + f'{DESC_TRANSITIONS}.csv'

NPY_URBAN_2025   = BASE_PATH + f'{P}urban2025.npy'
NPY_URBAN_2020   = BASE_PATH + f'{P}urban2020.npy'
NPY_SLOPE        = BASE_PATH + f'{P}slope.npy'
NPY_EXCLUSION    = BASE_PATH + f'{P}exclusion.npy'
NPY_SUIT_2025    = BASE_PATH + f'{P}suit2025.npy'
NPY_SUIT_2020    = BASE_PATH + f'{P}suit2020.npy'
PKL_PCA          = BASE_PATH + f'{P}pca.pkl'
PKL_RF           = BASE_PATH + f'{P}rf.pkl'
JSON_META        = BASE_PATH + f'{P}meta.json'
JSON_TIMESERIES  = BASE_PATH + f'{P}timeseries.json'
JSON_VALIDATION  = BASE_PATH + f'{P}validation.json'
TIF_HINDCAST     = BASE_PATH + f'{P}hindcast_2020_2025.tif'
TIF_BAU_2030     = BASE_PATH + f'{P}proj_BAU_2030.tif'
TIF_COMPACT_2030 = BASE_PATH + f'{P}proj_compact_2030.tif'
TIF_CLIMATE_2030 = BASE_PATH + f'{P}proj_climate_2030.tif'
GEOJSON_SUBCTY   = BASE_PATH + f'{P}subcounty_2030.geojson'

PIXEL_KM2     = 0.0001
ALL_YEARS     = list(range(2017, 2026))
HOLDOUT_YEAR  = 2020
GEE_PROJECT   = 'ee-emmanuelokellootieno'
AOI_ASSET     = 'projects/ee-emmanuelokellootieno/assets/Nairobi_Metropoli'
TRAINING_ASSET= 'projects/ee-emmanuelokellootieno/assets/Nairobi_Trg3'
EMB_COLLECTION= 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
CRS_EXPORT    = 'EPSG:32737'
# =============================================================================

ee.Initialize(project=GEE_PROJECT)
os.makedirs(BASE_PATH,  exist_ok=True)
os.makedirs(MASKS_PATH, exist_ok=True)

print(f"Part 1 setup complete.")

rm: cannot remove '/content/drive/MyDrive': Operation canceled
rm: cannot remove '/content/drive/Shareddrives': Operation canceled
Mounted at /content/drive
Part 1 setup complete.


In [ ]:
# =============================================================================
# NAIROBI URBAN GROWTH MODEL — AlphaEarth + CA   [v3]
# PART 2 OF 6: FILE READINESS CHECK + TRANSITION SAMPLING
#
# Run ONLY after all Part 1 GEE Tasks show green tick.
# What this does:
#   • Verifies every required file is present in Drive
#   • Rebuilds GEE objects and re-trains classifier (fast, same seed)
#   • Samples transition data across 8 year-pairs
#   • Exports balanced transition CSV to Drive
# =============================================================================

import ee
import os
import shutil
import numpy as np
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
ee.Authenticate()

# =============================================================================
# CONFIGURATION  —  paste unchanged into every part
# =============================================================================
VER           = 'v3'
P             = f'nrbca{VER}_'

DRIVE_BASE    = f'NrbCA_{VER}'
DRIVE_MASKS   = f'{DRIVE_BASE}/Masks'
BASE_PATH     = f'/content/drive/MyDrive/{DRIVE_BASE}/'
MASKS_PATH    = f'/content/drive/MyDrive/{DRIVE_MASKS}/'

DESC_MASK        = lambda y: f'{P}mask_{y}'
DESC_EMBED_2025  = f'{P}embed_2025'
DESC_SLOPE       = f'{P}slope'
DESC_EXCLUSION   = f'{P}exclusion'
DESC_TRANSITIONS = f'{P}transitions'

TIF_MASK         = lambda y: MASKS_PATH + f'{DESC_MASK(y)}.tif'
TIF_EMBED_2025   = BASE_PATH + f'{DESC_EMBED_2025}.tif'
TIF_SLOPE        = BASE_PATH + f'{DESC_SLOPE}.tif'
TIF_EXCLUSION    = BASE_PATH + f'{DESC_EXCLUSION}.tif'
CSV_TRANSITIONS  = BASE_PATH + f'{DESC_TRANSITIONS}.csv'

NPY_URBAN_2025   = BASE_PATH + f'{P}urban2025.npy'
NPY_URBAN_2020   = BASE_PATH + f'{P}urban2020.npy'
NPY_SLOPE        = BASE_PATH + f'{P}slope.npy'
NPY_EXCLUSION    = BASE_PATH + f'{P}exclusion.npy'
NPY_SUIT_2025    = BASE_PATH + f'{P}suit2025.npy'
NPY_SUIT_2020    = BASE_PATH + f'{P}suit2020.npy'
PKL_PCA          = BASE_PATH + f'{P}pca.pkl'
PKL_RF           = BASE_PATH + f'{P}rf.pkl'
JSON_META        = BASE_PATH + f'{P}meta.json'
JSON_TIMESERIES  = BASE_PATH + f'{P}timeseries.json'
JSON_VALIDATION  = BASE_PATH + f'{P}validation.json'
TIF_HINDCAST     = BASE_PATH + f'{P}hindcast_2020_2025.tif'
TIF_BAU_2030     = BASE_PATH + f'{P}proj_BAU_2030.tif'
TIF_COMPACT_2030 = BASE_PATH + f'{P}proj_compact_2030.tif'
TIF_CLIMATE_2030 = BASE_PATH + f'{P}proj_climate_2030.tif'
GEOJSON_SUBCTY   = BASE_PATH + f'{P}subcounty_2030.geojson'

PIXEL_KM2     = 0.0001
ALL_YEARS     = list(range(2017, 2026))
HOLDOUT_YEAR  = 2020
N_PER_CLASS   = 8000
GEE_PROJECT   = 'ee-emmanuelokellootieno'
AOI_ASSET     = 'projects/ee-emmanuelokellootieno/assets/Nairobi_Metropoli'
TRAINING_ASSET= 'projects/ee-emmanuelokellootieno/assets/Nairobi_Trg3'
EMB_COLLECTION= 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
CRS_EXPORT    = 'EPSG:32737'
# =============================================================================

ee.Initialize(project=GEE_PROJECT)

# =============================================================================
# STEP 1 — FILE READINESS CHECK
# =============================================================================
print("=" * 60)
print("STEP 1 — FILE READINESS CHECK")
print(f"Expecting files prefixed with '{P}' in MyDrive/{DRIVE_BASE}/")
print("=" * 60)

REQUIRED = {
    f'{DESC_SLOPE}.tif':      TIF_SLOPE,
    f'{DESC_EXCLUSION}.tif':  TIF_EXCLUSION,
}
for y in ALL_YEARS:
    k = f'{DESC_MASK(y)}.tif'
    REQUIRED[k] = TIF_MASK(y)

missing = []
for label, path in REQUIRED.items():
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1e6
        print(f"  OK      {label}  ({mb:.1f} MB)")
    else:
        print(f"  MISSING {label}")
        print(f"          {path}")
        missing.append((label, path))

# Special check for tiled embedding images
embed_files = [f for f in os.listdir(BASE_PATH) if f.startswith(f'{P}embed_2025-') and f.endswith('.tif')]
if not embed_files:
    missing.append((f'{DESC_EMBED_2025}.tif (tiled)', TIF_EMBED_2025)) # Add to missing if no tiles found
else:
    print(f"  OK      {DESC_EMBED_2025}.tif (tiled) ({len(embed_files)} files)")

if missing:
    print(f"\n{len(missing)} file(s) missing.")
    print("Check the GEE Tasks tab — exports may still be running.")
    print("\nAll TIF files currently in your Drive:")
    import subprocess
    r = subprocess.run(
        ['find', '/content/drive/MyDrive', '-name', '*.tif', '-type', 'f'],
        capture_output=True, text=True, timeout=90
    )
    for line in r.stdout.strip().split('\n')[:40]:
        if line:
            mb = os.path.getsize(line) / 1e6
            print(f"  {line}  ({mb:.1f} MB)")
    raise SystemExit(
        "\nDo NOT continue until all required files are present.\n"
        "Fix: wait for GEE Tasks to complete, then re-run Part 2."
    )

print(f"\nAll {len(REQUIRED) + (1 if embed_files else 0)} required files present.\n")

# =============================================================================
# STEP 2 — REBUILD GEE OBJECTS
# =============================================================================
print("Rebuilding GEE objects ...")

subcounties = ee.FeatureCollection(AOI_ASSET)
geometry    = subcounties.geometry()
emb_col     = ee.ImageCollection(EMB_COLLECTION)
training_fc = ee.FeatureCollection(TRAINING_ASSET)

year_img_2025 = (emb_col
                 .filterDate('2025-01-01', '2026-01-01')
                 .mosaic()
                 .clip(geometry))

band_names = year_img_2025.bandNames().getInfo()

sampled_pts = year_img_2025.sampleRegions(
    collection=training_fc,
    properties=['Landuse'],
    scale=10, tileScale=16
).randomColumn('random', 42)

rf_gee = (ee.Classifier
          .smileRandomForest(numberOfTrees=200, seed=42)
          .train(features=sampled_pts.filter(ee.Filter.lte('random', 0.7)),
                 classProperty='Landuse',
                 inputProperties=band_names))

year_images = {}
urban_masks = {}
for y in ALL_YEARS:
    img  = (emb_col
            .filterDate(f'{y}-01-01', f'{y+1}-01-01')
            .mosaic()
            .clip(geometry))
    mask = img.classify(rf_gee).eq(1).toByte().rename(f'urban_{y}')
    year_images[y] = img
    urban_masks[y] = mask

print("GEE objects rebuilt.\n")

# =============================================================================
# STEP 3 — TRANSITION SAMPLE GENERATION
#
# [F1] change_mask = non-urban → urban ONLY (not bidirectional)
# [F4] cosine_sim uses year-specific urban centroid (no future leakage)
# [F7] no-change pool = non-urban → non-urban (already-urban excluded)
# =============================================================================
print("=" * 60)
print("STEP 3 — TRANSITION SAMPLING")
print(f"  {N_PER_CLASS} samples per class per year-pair")
print(f"  Hold-out year {HOLDOUT_YEAR} excluded from training data")
print("=" * 60)

def get_transition_samples(year1, year2, n=N_PER_CLASS):
    img1  = year_images[year1]
    mask1 = urban_masks[year1]
    mask2 = urban_masks[year2]

    non_urban_t1 = mask1.eq(0)

    # [F1] Urbanisation only — not bidirectional
    pos_mask = non_urban_t1.And(mask2.eq(1))
    # [F7] No-change restricted to non-urban pixels
    neg_mask = non_urban_t1.And(mask2.eq(0))

    # [F4] Year-specific cosine similarity — no 2025 leakage
    avg_dict = (img1.updateMask(mask1)
                .reduceRegion(reducer=ee.Reducer.mean(),
                              geometry=geometry,
                              scale=30, maxPixels=1e9)
                .getInfo())
    avg_arr  = np.array([avg_dict.get(b, 0) for b in band_names])
    avg_img  = ee.Image.constant(avg_arr.tolist()).rename(band_names)
    dot      = img1.multiply(avg_img).reduce(ee.Reducer.sum())
    norm_img = img1.pow(2).reduce(ee.Reducer.sum()).sqrt()
    avg_norm = float(np.linalg.norm(avg_arr)) + 1e-8
    cosine   = dot.divide(norm_img.multiply(avg_norm)).rename('cosine_sim')

    feat_img = img1.addBands(cosine)

    def sample(mask, label):
        return (feat_img.updateMask(mask)
                .sample(region=geometry, scale=10, numPixels=n, seed=42)
                .map(lambda f: f.set('transition', label,
                                     'from_year',  year1,
                                     'to_year',    year2)))

    return sample(pos_mask, 1).merge(sample(neg_mask, 0))


all_fcs = []
for i in range(len(ALL_YEARS) - 1):
    y1, y2 = ALL_YEARS[i], ALL_YEARS[i + 1]
    if y1 == HOLDOUT_YEAR:
        print(f"  {y1}-{y2}: SKIPPED (hold-out year)")
        continue
    fc = get_transition_samples(y1, y2)
    n  = fc.size().getInfo()
    all_fcs.append(fc)
    print(f"  {y1}-{y2}: {n:,} samples")

merged = all_fcs[0]
for fc in all_fcs[1:]:
    merged = merged.merge(fc)

total = merged.size().getInfo()
print(f"\nTotal: {total:,} samples across {len(all_fcs)} year-pairs")

# =============================================================================
# STEP 4 — EXPORT TRANSITION CSV
# =============================================================================
print(f"\nExporting transition CSV as '{DESC_TRANSITIONS}' ...")

existing = {t.status()['description']
            for t in ee.batch.Task.list()
            if t.status()['state'] in ('COMPLETED', 'RUNNING', 'READY')}

if DESC_TRANSITIONS in existing:
    print(f"  Already submitted — skipping.")
else:
    ee.batch.Export.table.toDrive(
        collection=merged,
        description=DESC_TRANSITIONS,
        folder=DRIVE_BASE,
        fileFormat='CSV'
    ).start()
    print(f"  Submitted.")

print(f"""
=============================================================
PART 2 COMPLETE

CSV will appear as:
  MyDrive/{DRIVE_BASE}/{DESC_TRANSITIONS}.csv

Wait for CSV export to finish (GEE Tasks tab),
then run Part 3.
=============================================================
""")


Mounted at /content/drive
STEP 1 — FILE READINESS CHECK
Expecting files prefixed with 'nrbcav3_' in MyDrive/NrbCA_v3/
  MISSING nrbcav3_slope.tif
          /content/drive/MyDrive/NrbCA_v3/nrbcav3_slope.tif
  MISSING nrbcav3_exclusion.tif
          /content/drive/MyDrive/NrbCA_v3/nrbcav3_exclusion.tif
  MISSING nrbcav3_mask_2017.tif
          /content/drive/MyDrive/NrbCA_v3/Masks/nrbcav3_mask_2017.tif
  MISSING nrbcav3_mask_2018.tif
          /content/drive/MyDrive/NrbCA_v3/Masks/nrbcav3_mask_2018.tif
  MISSING nrbcav3_mask_2019.tif
          /content/drive/MyDrive/NrbCA_v3/Masks/nrbcav3_mask_2019.tif
  MISSING nrbcav3_mask_2020.tif
          /content/drive/MyDrive/NrbCA_v3/Masks/nrbcav3_mask_2020.tif
  MISSING nrbcav3_mask_2021.tif
          /content/drive/MyDrive/NrbCA_v3/Masks/nrbcav3_mask_2021.tif
  MISSING nrbcav3_mask_2022.tif
          /content/drive/MyDrive/NrbCA_v3/Masks/nrbcav3_mask_2022.tif
  MISSING nrbcav3_mask_2023.tif
          /content/drive/MyDrive/NrbCA_v3/Masks/nrbcav

SystemExit: 
Do NOT continue until all required files are present.
Fix: wait for GEE Tasks to complete, then re-run Part 2.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import os

if os.path.exists(CSV_TRANSITIONS):
    print(f"The file {CSV_TRANSITIONS} exists.")
else:
    print(f"The file {CSV_TRANSITIONS} does NOT exist.")

The file /content/drive/MyDrive/NrbCA_v3/nrbcav3_transitions.csv does NOT exist.


In [ ]:
# =============================================================================
# NAIROBI URBAN GROWTH MODEL — AlphaEarth + CA   [v3]
# PART 3 OF 6: LOCAL RF TRANSITION MODEL + PCA ANALYSIS
#
# Run after CSV export from Part 2 is complete.
# No GEE connection needed — fully local.
# Saves pca and rf models as .pkl for Parts 4–6 to load.
# =============================================================================

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# =============================================================================
# CONFIGURATION  —  paste unchanged into every part
# =============================================================================
VER           = 'v3'
P             = f'nrbca{VER}_'

DRIVE_BASE    = f'NrbCA_{VER}'
DRIVE_MASKS   = f'{DRIVE_BASE}/Masks'
BASE_PATH     = f'/content/drive/MyDrive/{DRIVE_BASE}/'
MASKS_PATH    = f'/content/drive/MyDrive/{DRIVE_MASKS}/'

DESC_MASK        = lambda y: f'{P}mask_{y}'
DESC_EMBED_2025  = f'{P}embed_2025'
DESC_SLOPE       = f'{P}slope'
DESC_EXCLUSION   = f'{P}exclusion'
DESC_TRANSITIONS = f'{P}transitions'

TIF_MASK         = lambda y: MASKS_PATH + f'{DESC_MASK(y)}.tif'
TIF_EMBED_2025   = BASE_PATH + f'{DESC_EMBED_2025}.tif'
TIF_SLOPE        = BASE_PATH + f'{DESC_SLOPE}.tif'
TIF_EXCLUSION    = BASE_PATH + f'{DESC_EXCLUSION}.tif'
CSV_TRANSITIONS  = BASE_PATH + f'{DESC_TRANSITIONS}.csv'

NPY_URBAN_2025   = BASE_PATH + f'{P}urban2025.npy'
NPY_URBAN_2020   = BASE_PATH + f'{P}urban2020.npy'
NPY_SLOPE        = BASE_PATH + f'{P}slope.npy'
NPY_EXCLUSION    = BASE_PATH + f'{P}exclusion.npy'
NPY_SUIT_2025    = BASE_PATH + f'{P}suit2025.npy'
NPY_SUIT_2020    = BASE_PATH + f'{P}suit2020.npy'
PKL_PCA          = BASE_PATH + f'{P}pca.pkl'
PKL_RF           = BASE_PATH + f'{P}rf.pkl'
JSON_META        = BASE_PATH + f'{P}meta.json'
JSON_TIMESERIES  = BASE_PATH + f'{P}timeseries.json'
JSON_VALIDATION  = BASE_PATH + f'{P}validation.json'
TIF_HINDCAST     = BASE_PATH + f'{P}hindcast_2020_2025.tif'
TIF_BAU_2030     = BASE_PATH + f'{P}proj_BAU_2030.tif'
TIF_COMPACT_2030 = BASE_PATH + f'{P}proj_compact_2030.tif'
TIF_CLIMATE_2030 = BASE_PATH + f'{P}proj_climate_2030.tif'
GEOJSON_SUBCTY   = BASE_PATH + f'{P}subcounty_2030.geojson'

PIXEL_KM2     = 0.0001
ALL_YEARS     = list(range(2017, 2026))
HOLDOUT_YEAR  = 2020
GEE_PROJECT   = 'ee-emmanuelokellootieno'
AOI_ASSET     = 'projects/ee-emmanuelokellootieno/assets/Nairobi_Metropoli'
TRAINING_ASSET= 'projects/ee-emmanuelokellootieno/assets/Nairobi_Trg3'
EMB_COLLECTION= 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
CRS_EXPORT    = 'EPSG:32737'
FIG_LOADINGS_PC2 = BASE_PATH + f'{P}fig_loadings_PC2.png'
FIG_LOADINGS_PC3 = BASE_PATH + f'{P}fig_loadings_PC3.png'
FIG_PCA_HEATMAP  = BASE_PATH + f'{P}fig_pca_heatmap.png'
FIG_RF_IMP       = BASE_PATH + f'{P}fig_rf_importance.png'
# =============================================================================

# =============================================================================
# STEP 1 — LOAD CSV
# =============================================================================
print("=" * 60)
print("STEP 1 — LOADING TRANSITION DATA")
print("=" * 60)

if not os.path.exists(CSV_TRANSITIONS):
    raise FileNotFoundError(
        f"\nCSV not found: {CSV_TRANSITIONS}\n"
        "GEE export from Part 2 may still be running.\n"
        "Check Tasks tab in Earth Engine Code Editor."
    )

df = pd.read_csv(CSV_TRANSITIONS)
print(f"Rows: {len(df):,}")

embed_cols = [c for c in df.columns if c.startswith('A')]
has_cosine = 'cosine_sim' in df.columns
y_all      = df['transition'].values.astype(int)
counts     = np.bincount(y_all)

print(f"AlphaEarth bands in CSV : {len(embed_cols)}  (expected 64)")
print(f"Cosine similarity column: {has_cosine}")
print(f"\nClass distribution:")
print(f"  No-change (0): {counts[0]:,}  ({100*counts[0]/len(y_all):.1f}%)")
print(f"  Urbanised (1): {counts[1]:,}  ({100*counts[1]/len(y_all):.1f}%)")

if len(embed_cols) != 64:
    print(f"\nWARNING: expected 64 embedding columns, found {len(embed_cols)}.")
    print("Check that the CSV exported cleanly from GEE.")

# =============================================================================
# STEP 2 — PCA  [F2: float64]
# =============================================================================
print("\n" + "=" * 60)
print("STEP 2 — PCA DIMENSIONALITY REDUCTION")
print("=" * 60)

X_embed = df[embed_cols].values.astype(np.float64)   # [F2] float64
pca     = PCA(n_components=15, random_state=42)
X_pca   = pca.fit_transform(X_embed)

cumvar  = np.cumsum(pca.explained_variance_ratio_)
print(f"Total variance explained by 15 PCs: {cumvar[-1]:.1%}")
for i, v in enumerate(cumvar[:8]):
    print(f"  PC{i+1:2d}: {pca.explained_variance_ratio_[i]:.3%}  "
          f"(cumulative {v:.1%})")

# =============================================================================
# STEP 3 — FEATURE MATRIX
# =============================================================================
if has_cosine:
    X          = np.hstack([X_pca, df[['cosine_sim']].values])
    feat_names = [f'PC{i+1}' for i in range(15)] + ['cosine_sim']
else:
    X          = X_pca
    feat_names = [f'PC{i+1}' for i in range(15)]
    print("cosine_sim not in CSV — using PCA features only.")

print(f"\nFeature matrix: {X.shape}")

# =============================================================================
# STEP 4 — STRATIFIED 5-FOLD CV  [F5: class_weight='balanced']
# =============================================================================
print("\n" + "=" * 60)
print("STEP 4 — CROSS-VALIDATION  (5-fold stratified)")
print("  Note: spatial block CV recommended for publication")
print("=" * 60)

cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_cv = RandomForestClassifier(
    n_estimators=300, max_depth=20, n_jobs=-1,
    random_state=42, class_weight='balanced'
)
for metric in ('f1', 'roc_auc', 'accuracy'):
    scores = cross_val_score(rf_cv, X, y_all, cv=cv,
                             scoring=metric, n_jobs=-1)
    print(f"  {metric:10s}: {scores.mean():.3f} ± {scores.std():.3f}")

# =============================================================================
# STEP 5 — FINAL MODEL  [F5: class_weight='balanced']
# =============================================================================
print("\n" + "=" * 60)
print("STEP 5 — TRAIN FINAL RF TRANSITION MODEL")
print("=" * 60)

rf_trans = RandomForestClassifier(
    n_estimators=300, max_depth=20, n_jobs=-1,
    random_state=42, class_weight='balanced'
)
rf_trans.fit(X, y_all)

print(classification_report(y_all, rf_trans.predict(X),
                             target_names=['no-change', 'urbanised']))

imp = pd.Series(rf_trans.feature_importances_, index=feat_names)
print("Top 10 importances:")
print(imp.nlargest(10).round(5).to_string())

# =============================================================================
# STEP 6 — SAVE MODELS
# =============================================================================
joblib.dump(pca,      PKL_PCA)
joblib.dump(rf_trans, PKL_RF)
print(f"\nModels saved:")
print(f"  {PKL_PCA}")
print(f"  {PKL_RF}")

# =============================================================================
# STEP 7 — PCA LOADING PLOTS
# =============================================================================
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(15)],
    index=embed_cols
)

for pc, path in [('PC2', FIG_LOADINGS_PC2), ('PC3', FIG_LOADINGS_PC3)]:
    top    = loadings[pc].sort_values(key=abs, ascending=False)[:20]
    colors = ['#1D9E75' if v > 0 else '#D85A30' for v in top]
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(top.index, top.values, color=colors)
    ax.axvline(0, color='black', lw=0.8, ls='--')
    ax.set_title(f'PCA Loadings — {pc} (Top 20 AlphaEarth dims)')
    ax.set_xlabel('Loading value')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"  Saved: {path.split('/')[-1]}")

top20 = loadings.abs().max(axis=1).nlargest(20).index
fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(loadings.loc[top20], cmap='RdBu_r', center=0,
            linewidths=0.3, vmin=-0.5, vmax=0.5, ax=ax)
ax.set_title('AlphaEarth embedding loadings — top 20 dims × 15 PCs')
plt.tight_layout()
plt.savefig(FIG_PCA_HEATMAP, dpi=150)
plt.show()
print(f"  Saved: {FIG_PCA_HEATMAP.split('/')[-1]}")

fig, ax = plt.subplots(figsize=(10, 5))
imp.nlargest(15).sort_values().plot(kind='barh', ax=ax, color='#378ADD')
ax.set_title('RF feature importances (top 15)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(FIG_RF_IMP, dpi=150)
plt.show()
print(f"  Saved: {FIG_RF_IMP.split('/')[-1]}")

print(f"""
=============================================================
PART 3 COMPLETE

Saved to MyDrive/{DRIVE_BASE}/:
  {P}pca.pkl
  {P}rf.pkl
  {P}fig_loadings_PC2.png
  {P}fig_loadings_PC3.png
  {P}fig_pca_heatmap.png
  {P}fig_rf_importance.png

Proceed to Part 4.
=============================================================
""")


In [ ]:
# =============================================================================
# NAIROBI URBAN GROWTH MODEL — AlphaEarth + CA   [v3]
# PART 4 OF 6: LOAD RASTERS, GROWTH RATE & SUITABILITY MAP
#
# Run after Part 3. No GEE connection needed.
# Loads the 64-band embedding TIF and generates a pixel-level
# RF transition probability probability surface using tiled inference.
# Expected runtime: 20–45 min (depends on Colab RAM tier).
# =============================================================================

# Upgrade pip to ensure latest resolver capabilities
!pip install --upgrade pip --quiet

# Fix rasterio installation to a known working version and install GDAL tools
!pip uninstall -y rasterio --quiet # Uninstall existing rasterio
!pip install "rasterio==1.5.0" --quiet --force-reinstall # Install a specific working version
!sudo apt-get update --quiet
!sudo apt-get install gdal-bin --quiet # Install GDAL command-line tools

# Resolve numpy conflicts:
# First, uninstall tensorflow if it's the primary cause of numpy version conflicts
# Note: This assumes tensorflow is not strictly required for the notebook's core functionality.
!pip uninstall -y tensorflow --quiet

# Aggressively uninstall numpy, scipy, scikit-image, numba to ensure clean state
!pip uninstall -y numpy scipy scikit-image numba --quiet

# Then, ensure a compatible numpy version is installed, e.g., 1.26.0 which is compatible with Python 3.12 and Numba 0.60.0
!pip install numpy==1.26.0 --quiet --force-reinstall # Install compatible numpy and ensure it's used

# Reinstall scipy and scikit-image with versions known to work with numpy 1.x and Python 3.12
!pip install scipy==1.12.0 scikit-image==0.22.0 --quiet

import os
import gc
import json
import numpy as np
import rasterio
from rasterio.windows import Window
from skimage.transform import resize
import joblib
from google.colab import drive
import tempfile
import subprocess # Added for gdalbuildvrt
# import rasterio.vrt # No longer needed if using subprocess for VRT

# Add diagnostic prints to confirm versions
import sys
print(f"Python version: {sys.version}")
print(f"Numpy version after installs: {np.__version__}")
import scipy
import skimage
print(f"Scipy version after installs: {scipy.__version__}")
print(f"Scikit-image version after installs: {skimage.__version__}")

drive.mount('/content/drive', force_remount=True)

# =============================================================================
# CONFIGURATION  —  paste unchanged into every part
# =============================================================================
VER           = 'v3'
P             = f'nrbca{VER}_'

DRIVE_BASE    = f'NrbCA_{VER}'
DRIVE_MASKS   = f'{DRIVE_BASE}/Masks'
BASE_PATH     = f'/content/drive/MyDrive/{DRIVE_BASE}/'
MASKS_PATH    = f'/content/drive/MyDrive/{DRIVE_MASKS}/'

DESC_MASK        = lambda y: f'{P}mask_{y}'
DESC_EMBED_2025  = f'{P}embed_2025'
DESC_SLOPE       = f'{P}slope'
DESC_EXCLUSION   = f'{P}exclusion'
DESC_TRANSITIONS = f'{P}transitions'

TIF_MASK         = lambda y: MASKS_PATH + f'{DESC_MASK(y)}.tif'
TIF_EMBED_2025   = BASE_PATH + f'{DESC_EMBED_2025}.tif'
TIF_SLOPE        = BASE_PATH + f'{DESC_SLOPE}.tif'
TIF_EXCLUSION    = BASE_PATH + f'{DESC_EXCLUSION}.tif'
CSV_TRANSITIONS  = BASE_PATH + f'{DESC_TRANSITIONS}.csv'

NPY_URBAN_2025   = BASE_PATH + f'{P}urban2025.npy'
NPY_URBAN_2020   = BASE_PATH + f'{P}urban2020.npy'
NPY_SLOPE        = BASE_PATH + f'{P}slope.npy'
NPY_EXCLUSION    = BASE_PATH + f'{P}exclusion.npy'
NPY_SUIT_2025    = BASE_PATH + f'{P}suit2025.npy'
NPY_SUIT_2020    = BASE_PATH + f'{P}suit2020.npy'
PKL_PCA          = BASE_PATH + f'{P}pca.pkl'
PKL_RF           = BASE_PATH + f'{P}rf.pkl'
JSON_META        = BASE_PATH + f'{P}meta.json'
JSON_TIMESERIES  = BASE_PATH + f'{P}timeseries.json'
JSON_VALIDATION  = BASE_PATH + f'{P}validation.json'
TIF_HINDCAST     = BASE_PATH + f'{P}hindcast_2020_2025.tif'
TIF_BAU_2030     = BASE_PATH + f'{P}proj_BAU_2030.tif'
TIF_COMPACT_2030 = BASE_PATH + f'{P}proj_compact_2030.tif'
TIF_CLIMATE_2030 = BASE_PATH + f'{P}proj_climate_2030.tif'
GEOJSON_SUBCTY   = BASE_PATH + f'{P}subcounty_2030.geojson'

PIXEL_KM2     = 0.0001
ALL_YEARS     = list(range(2017, 2026))
HOLDOUT_YEAR  = 2020
GEE_PROJECT   = 'ee-emmanuelokellootieno'
AOI_ASSET     = 'projects/ee-emmanuelokellootieno/assets/Nairobi_Metropoli'
TRAINING_ASSET= 'projects/ee-emmanuelokellootieno/assets/Nairobi_Trg3'
EMB_COLLECTION= 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
CRS_EXPORT    = 'EPSG:32737'
# =============================================================================

# =============================================================================
# STEP 1 — LOAD RASTERS
# No fallback paths needed — all files are uniquely named with prefix P
# =============================================================================
print("—" * 60)
print("STEP 1 — LOADING RASTERS")
print("—" * 60)

def must_exist(path, label):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\nMISSING: {label}\n"
            f"Expected: {path}\n"
            "Ensure Part 1 GEE exports completed successfully."
        )
    mb = os.path.getsize(path) / 1e6
    print(f"  OK  {label}  ({mb:.1f} MB)")
    return path

must_exist(TIF_SLOPE,       f'{DESC_SLOPE}.tif')
must_exist(TIF_EXCLUSION,   f'{DESC_EXCLUSION}.tif')

# Check for tiled embedding files
embed_files = [os.path.join(BASE_PATH, f) for f in os.listdir(BASE_PATH) if f.startswith(f'{P}embed_2025-') and f.endswith('.tif')]
if not embed_files:
    raise FileNotFoundError(
        f"\nMISSING: {DESC_EMBED_2025}.tif (tiled)\n"
        f"Expected tiled embedding files in: {BASE_PATH}\n"
        "Ensure Part 1 GEE exports completed successfully."
    )
else:
    print(f"  OK  {DESC_EMBED_2025}.tif (tiled) ({len(embed_files)} files)")

must_exist(TIF_MASK(2025),  f'{DESC_MASK(2025)}.tif')

# — Urban 2025 ————————————————————————————————————————————————————————————
with rasterio.open(TIF_MASK(2025)) as src:
    urban2025 = src.read(1).astype(np.uint8)
    transform = src.transform
    crs       = src.crs
    shape     = urban2025.shape

# — Slope —————————————————————————————————————————————————————————————————
with rasterio.open(TIF_SLOPE) as src:
    raw   = src.read(1)
    slope = (resize(raw, shape, order=1, preserve_range=True).astype(np.float32)
             if raw.shape != shape else raw.astype(np.float32))
    del raw

# — Exclusion —————————————————————————————————————————————————————————————
with rasterio.open(TIF_EXCLUSION) as src:
    raw       = src.read(1)
    exclusion = (resize(raw, shape, order=0, preserve_range=True).astype(np.uint8)
                 if raw.shape != shape else raw.astype(np.uint8))
    del raw

gc.collect()
print(f"\n  Shape:      {shape}")
print(f"  Urban 2025: {urban2025.sum() * PIXEL_KM2:.1f} km²")
print(f"  Excluded:   {exclusion.sum() * PIXEL_KM2:.1f} km²  (slope > 25°)")

# — Hold-out mask 2020 ————————————————————————————————————————————————————
has_2020 = os.path.exists(TIF_MASK(HOLDOUT_YEAR))
if has_2020:
    with rasterio.open(TIF_MASK(HOLDOUT_YEAR)) as src:
        raw       = src.read(1).astype(np.uint8)
        urban2020 = (resize(raw, shape, order=0, preserve_range=True).astype(np.uint8)
                     if raw.shape != shape else raw)
        del raw
    print(f"  Urban 2020: {urban2020.sum() * PIXEL_KM2:.1f} km²  (hold-out)")
else:
    urban2020 = None
    print(f"  WARNING: {TIF_MASK(HOLDOUT_YEAR)} not found — FoM validation will be skipped.")

# =============================================================================
# STEP 2 — DERIVE GROWTH RATE FROM OBSERVED MASKS
# =============================================================================
print("\n" + "=" * 60)
print("STEP 2 — HISTORICAL GROWTH RATE")
print("—" * 60)

hist_years, hist_areas = [], []
for y in sorted(ALL_YEARS):
    path = TIF_MASK(y)
    if os.path.exists(path):
        with rasterio.open(path) as src:
            area = src.read(1).sum() * PIXEL_KM2
        hist_years.append(y)
        hist_areas.append(area)
        print(f"  {y}: {area:.1f} km²")
    else:
        print(f"  {y}: not found — skipped")

if len(hist_years) >= 3:
    growth_rates   = np.diff(hist_areas)
    AVG_GROWTH_KM2 = float(np.nanmean(growth_rates))
else:
    AVG_GROWTH_KM2 = 22.78
    print("  Insufficient masks — using default 22.78 km²/yr")

print(f"\n  Average growth rate: {AVG_GROWTH_KM2:.3f} km²/yr")

# =============================================================================
# STEP 3 — LOAD MODELS
# =============================================================================
print("\n" + "=" * 60)
print("STEP 3 — LOADING MODELS FROM PART 3")
print("—" * 60)

for p in (PKL_PCA, PKL_RF):
    must_exist(p, p.split('/')[-1])

pca      = joblib.load(PKL_PCA)
rf_trans = joblib.load(PKL_RF)
print(f"  PCA: {pca.n_components_} components")
print(f"  RF : {rf_trans.n_estimators} trees, "
      f"class_weight={rf_trans.class_weight}")

# =============================================================================
# STEP 4 — TILED SUITABILITY MAP  [F2: float32]  [F6: pure RF alpha=1.0]
# =============================================================================
print("\n" + "=" * 60)
print("STEP 4 — GENERATING RF SUITABILITY MAP")
print(f"  Embedding TIF: {DESC_EMBED_2025}.tif (tiled)")
print("  Using float32 blocks (F2), alpha=1.0 pure RF (F6)")
print("—" * 60)

def generate_suitability(embed_tif_paths, pca_model, rf_model,
                          shape, block=512):
    """
    Tiled RF transition probability probability map.
    [F2] float32 throughout — avoids float16 precision loss.
    [F6] Returns pure RF probability — no unjustified distance blend.
    """
    suit      = np.zeros(shape, dtype=np.float32)
    n_r, n_c = shape

    # Create a temporary VRT file
    with tempfile.NamedTemporaryFile(suffix=".vrt", delete=False) as tmp_vrt:
        vrt_path = tmp_vrt.name

    try:
        # Build the VRT using gdalbuildvrt
        vrt_cmd = ['gdalbuildvrt', '-r', 'nearest', vrt_path] + embed_tif_paths
        result = subprocess.run(vrt_cmd, capture_output=True, text=True, check=True)
        if result.stderr:
            print(f"gdalbuildvrt stderr: {result.stderr}")
        if result.stdout:
            print(f"gdalbuildvrt stdout: {result.stdout}")

        # Open the VRT as a single virtual dataset
        with rasterio.open(vrt_path) as src:
            n_bands    = src.count
            total_blks = ((n_r + block - 1) // block) * \
                         ((n_c + block - 1) // block)
            done       = 0

            for row in range(0, n_r, block):
                for col in range(0, n_c, block):
                    h   = min(block, n_r - row)
                    w   = min(block, n_c - col)
                    win = Window(col, row, w, h)

                    # [F2] float32 — NOT float16
                    data = src.read(window=win).astype(np.float32)
                    data = data.reshape(n_bands, -1).T
                    data = np.nan_to_num(data, nan=0.0)

                    pca_blk = pca_model.transform(data)        # (N, 15)
                    proba   = rf_model.predict_proba(pca_blk)[:, 1]
                    suit[row:row+h, col:col+w] = proba.reshape(h, w)

                    del data, pca_blk, proba
                    done += 1
                    if done % 20 == 0 or done == total_blks:
                        print(f"  {100*done/total_blks:.0f}%  "
                              f"(row {min(row+block, n_r)}/{n_r})")
                    gc.collect()
    finally:
        # Clean up the temporary VRT file
        if os.path.exists(vrt_path):
            os.remove(vrt_path)

    return suit


print("\nGenerating suitability for 2025 (forward simulation start) ...")
suit_2025 = generate_suitability(embed_files, pca, rf_trans, shape)
np.save(NPY_SUIT_2025, suit_2025)
print(f"  Saved: {NPY_SUIT_2025.split('/')[-1]}")
print(f"  Mean: {suit_2025.mean():.4f}  Max: {suit_2025.max():.4f}")

if has_2020 and urban2020 is not None:
    print("\nGenerating suitability for 2020 (hold-out validation) ...")
    # Same embedding TIF — suitability surface is from embedding,
    # start mask differs in the CA loop
    suit_2020 = generate_suitability(embed_files, pca, rf_trans, shape)
    np.save(NPY_SUIT_2020, suit_2020)
    print(f"  Saved: {NPY_SUIT_2020.split('/')[-1]}")
else:
    suit_2020 = None

# =============================================================================
# STEP 5 — SAVE ALL ARRAYS AND METADATA FOR PART 5
# =============================================================================
np.save(NPY_URBAN_2025, urban2025)
np.save(NPY_SLOPE,      slope)
np.save(NPY_EXCLUSION,  exclusion)
if has_2020 and urban2020 is not None:
    np.save(NPY_URBAN_2020, urban2020)

meta = {
    'shape':          list(shape),
    'AVG_GROWTH_KM2': AVG_GROWTH_KM2,
    'PIXEL_KM2':      PIXEL_KM2,
    'crs':            str(crs),
    'transform':      list(transform)[:6],
    'hist_years':     hist_years,
    'hist_areas':     hist_areas,
    'has_2020':       has_2020 and urban2020 is not None,
    'version':        VER,
}
with open(JSON_META, 'w') as f:
    json.dump(meta, f, indent=2)

print(f"\nMetadata saved: {JSON_META.split('/')[-1]}")

print(f"""
=============================================================
PART 4 COMPLETE

Saved to MyDrive/{DRIVE_BASE}/
  {P}urban2025.npy
  {P}urban2020.npy  (if available)
  {P}slope.npy
  {P}exclusion.npy
  {P}suit2025.npy
  {P}suit2020.npy   (if available)
  {P}meta.json

Proceed to Part 5 (CA simulation + FoM validation).
=============================================================
""")


In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Drive unmounted successfully.")

In [ ]:
import rasterio
print(f"Rasterio version: {rasterio.__version__}")

In [ ]:
import os

# List files in the Masks directory
print(f"Listing files in {os.getenv('HOME')}/content/drive/MyDrive/NrbCA_v3/Masks/")
!ls -lh /content/drive/MyDrive/NrbCA_v3/Masks/

In [ ]:
import os

print(f"Listing files in {os.getenv('HOME')}/content/drive/MyDrive/NrbCA_v3/Masks/")
!ls -lh /content/drive/MyDrive/NrbCA_v3/Masks/

In [ ]:
import subprocess
import os

print("Searching for all .tif files in /content/drive/MyDrive/ (this may take a while)...")

r = subprocess.run(
    ['find', '/content/drive/MyDrive', '-name', '*.tif', '-type', 'f'],
    capture_output=True, text=True, timeout=300 # Increased timeout to 5 minutes for potentially large searches
)

if r.returncode == 0:
    found_tifs = r.stdout.strip().split('\n')
    if found_tifs and found_tifs[0]: # Check if list is not empty and first element is not empty
        print(f"Found {len(found_tifs)} .tif files:")
        for tif_path in found_tifs:
            try:
                mb = os.path.getsize(tif_path) / (1024 * 1024) # Convert bytes to MB
                print(f"  {tif_path} ({mb:.2f} MB)")
            except FileNotFoundError:
                print(f"  {tif_path} (File not found during size check)")
            except Exception as e:
                print(f"  {tif_path} (Error getting size: {e})")
    else:
        print("No .tif files found in /content/drive/MyDrive/")
else:
    print(f"Error searching for .tif files: {r.stderr}")

In [ ]:
import os
import shutil

# Define the incorrect and correct paths
INCORRECT_MASKS_DIR = '/content/drive/MyDrive/NrbCA_v3 Masks/'
CORRECT_BASE_DIR = '/content/drive/MyDrive/NrbCA_v3/'
CORRECT_MASKS_SUBDIR = os.path.join(CORRECT_BASE_DIR, 'Masks')

print(f"Attempting to move mask files from '{INCORRECT_MASKS_DIR}' to '{CORRECT_MASKS_SUBDIR}'...")

# 1. Ensure the correct base directory exists
os.makedirs(CORRECT_BASE_DIR, exist_ok=True)

# 2. Ensure the correct subfolder exists
os.makedirs(CORRECT_MASKS_SUBDIR, exist_ok=True)

# 3. Move all files from the incorrect directory to the correct one
if os.path.exists(INCORRECT_MASKS_DIR):
    files_moved = 0
    for filename in os.listdir(INCORRECT_MASKS_DIR):
        source_path = os.path.join(INCORRECT_MASKS_DIR, filename)
        destination_path = os.path.join(CORRECT_MASKS_SUBDIR, filename)
        if os.path.isfile(source_path): # Only move files
            shutil.move(source_path, destination_path)
            files_moved += 1
    print(f"Moved {files_moved} files from '{INCORRECT_MASKS_DIR}' to '{CORRECT_MASKS_SUBDIR}'.")

    # 4. Remove the now empty incorrect directory
    if not os.listdir(INCORRECT_MASKS_DIR): # Check if directory is empty before removing
        os.rmdir(INCORRECT_MASKS_DIR)
        print(f"Removed empty directory '{INCORRECT_MASKS_DIR}'.")
    else:
        print(f"Directory '{INCORRECT_MASKS_DIR}' was not empty after move, not removing.")
else:
    print(f"Source directory '{INCORRECT_MASKS_DIR}' does not exist. No files to move.")

print("Mask folder correction attempt complete.")


In [ ]:
import os

print("Listing contents of Google Drive root:")
!ls -F '/content/drive/MyDrive/'